# Module 3 – Building Semantic Search from Scratch
### Course: OpenAI Embeddings API | Pluralsight

---
## What You'll Learn
- Similarity metrics: cosine similarity, dot product, Euclidean distance
- Building an in-memory search index with numpy
- Re-ranking strategies for better relevance
- Hybrid search: BM25 keyword scoring + semantic embeddings
- Evaluating search quality: precision@k, recall@k, MRR

---
https://www.pinecone.io/learn/vector-similarity/

https://developers.openai.com/api/docs/guides/embeddings

In [28]:
# ─ SETUP · 1 — Install dependencies ────────────────────────────────
# !pip install openai numpy scikit-learn rank-bm25

In [29]:
# ─ SETUP · 2 — Imports & load data ─────────────────────────────────
import getpass
import os
import json
import numpy as np
from openai import OpenAI
from rank_bm25 import BM25Okapi

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key: ")
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

with open("../module2/yelp_embeddings.json") as f:
    embedded_reviews = json.load(f)

print(f"Loaded {len(embedded_reviews)} embedded reviews")
print(f"Vector dimensions: {len(embedded_reviews[0]['embedding'])}")

Loaded 500 embedded reviews
Vector dimensions: 1536


---
## Clip 1: Similarity Metrics and Search Architecture

**The whole architecture, in four steps:**
1. Embed the corpus (done back in Module 2)
2. Embed the query
3. Compute similarity between the query and every document
4. Return the top results

| Metric | Formula | Best when |
|---|---|---|
| **Cosine** | `dot(a, b) / (‖a‖ · ‖b‖)` | Normalized vectors — the default for text embeddings |
| **Dot product** | `dot(a, b)` | Magnitude carries meaning |
| **Euclidean** | `‖a − b‖` | Absolute distance matters (lower = more similar) |

In [30]:
# ─ CLIP 1 · 1 — The three similarity metrics ───────────────────────
def cosine_similarity(a: list[float], b: list[float]) -> float:
    """Direction-only similarity in [-1, 1]. Higher = more similar."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def dot_product(a: list[float], b: list[float]) -> float:
    """Like cosine but NOT normalized, so magnitude still counts."""
    return float(np.dot(np.array(a), np.array(b)))


def euclidean_distance(a: list[float], b: list[float]) -> float:
    """Straight-line distance. Lower = more similar."""
    return float(np.linalg.norm(np.array(a) - np.array(b)))

In [31]:
# ─ CLIP 1 · 2 — The reviews we'll compare ──────────────────────────
# We hand-picked three reviews by scanning the Yelp corpus:
#   reviews 40 & 41 are the SAME diner (Gab n Eat); review 62 is a Chinese takeout.
for i in [40, 41, 62]:
    r = embedded_reviews[i]
    print(f"Review {i}  ({r['stars']}*):  {r['text'][:95]}...\n")


Review 40  (5*):  I always love a good diner.  Gab and Eat was just what we were looking for on a Saturday mornin...

Review 41  (5*):  I have been coming to Gab n Eat for almost 20 years and They have never let me down. I get a ty...

Review 62  (2*):  Far away from real Chinese food. Doesn't even taste good as American style Chinese food....



In [32]:
# ─ CLIP 1 · 3 — Seeing it on real reviews ──────────────────────────
review_a = embedded_reviews[40]  # Gab n Eat diner - happy review
review_b = embedded_reviews[41]  # Gab n Eat diner - another happy review (SAME place)
review_c = embedded_reviews[62]  # a Chinese-takeout review (different place)

def compare_pair(label, r1, r2):
    cos = cosine_similarity(r1["embedding"], r2["embedding"])
    dot = dot_product(r1["embedding"], r2["embedding"])
    euc = euclidean_distance(r1["embedding"], r2["embedding"])
    print(f"\n{label}")
    print(f"  A: {r1['text'][:60]}...")
    print(f"  B: {r2['text'][:60]}...")
    print(f"  Cosine Similarity: {cos:.4f}  (higher = more similar)")
    print(f"  Dot Product:       {dot:.4f}")
    print(f"  Euclidean Dist:    {euc:.4f}  (lower = more similar)")

compare_pair("Pair A-B (two reviews of the SAME diner)", review_a, review_b)
compare_pair("Pair A-C (diner vs Chinese takeout - different places)", review_a, review_c)



Pair A-B (two reviews of the SAME diner)
  A: I always love a good diner.  Gab and Eat was just what we we...
  B: I have been coming to Gab n Eat for almost 20 years and They...
  Cosine Similarity: 0.6523  (higher = more similar)
  Dot Product:       0.6524
  Euclidean Dist:    0.8339  (lower = more similar)

Pair A-C (diner vs Chinese takeout - different places)
  A: I always love a good diner.  Gab and Eat was just what we we...
  B: Far away from real Chinese food. Doesn't even taste good as ...
  Cosine Similarity: 0.1820  (higher = more similar)
  Dot Product:       0.1820
  Euclidean Dist:    1.2794  (lower = more similar)


### CLIP 1 · 4 — Why we vectorize

`cosine_similarity` above compares **one pair** at a time. To search, we need the query
scored against **all 500** reviews — and in production, millions. A Python `for` loop is
far too slow, so we *vectorize*: normalize everything once, then let a single matrix
multiply score every document at the same time.

In [33]:
# ─ CLIP 1 · 5 — The vectorized engine ──────────────────────────────
def cosine_similarity_matrix(query_vec: np.ndarray, corpus_matrix: np.ndarray) -> np.ndarray:
    """
    Cosine similarity between ONE query vector and EVERY corpus vector.
        query_vec:     shape (dim,)
        corpus_matrix: shape (N, dim)
        returns:       shape (N,)   one score per document
    """
    # 1. Make the query a unit vector (length 1)
    query_norm = query_vec / np.linalg.norm(query_vec)
    # 2. Make every corpus row a unit vector
    corpus_norms = np.linalg.norm(corpus_matrix, axis=1, keepdims=True)  # (N, 1)
    corpus_normalized = corpus_matrix / corpus_norms                     # (N, dim)
    # 3. Dot product of unit vectors IS cosine similarity - all N at once
    return corpus_normalized @ query_norm                               # (N,)


# Sanity check: the vectorized version must agree with the per-pair version.
demo_corpus = np.array([r["embedding"] for r in embedded_reviews[40:43]])  # diners 40, 41, 42
demo_query  = np.array(embedded_reviews[40]["embedding"])                  # diner 40
print("Vectorized (diner 40 vs diners 40,41,42):", np.round(cosine_similarity_matrix(demo_query, demo_corpus), 4))
print("Pairwise   diner 40 vs 40:", round(cosine_similarity(embedded_reviews[40]["embedding"], embedded_reviews[40]["embedding"]), 4))
print("Pairwise   diner 40 vs 41:", round(cosine_similarity(embedded_reviews[40]["embedding"], embedded_reviews[41]["embedding"]), 4))


Vectorized (diner 40 vs diners 40,41,42): [1.     0.6523 0.6532]
Pairwise   diner 40 vs 40: 1.0
Pairwise   diner 40 vs 41: 0.6523


---
## Clip 2: Building and Querying a Search Index

In [34]:
# ─ CLIP 2 · 1 — Build the index ────────────────────────────────────
# The "index" is just two parallel structures: a matrix of vectors + a list of metadata.
corpus_matrix = np.array([r["embedding"] for r in embedded_reviews])  # (N, 1536)
metadata = [{"id": r["id"], "text": r["text"], "stars": r["stars"]} for r in embedded_reviews]

print(f"Index built: {corpus_matrix.shape}  ({corpus_matrix.shape[0]} reviews x {corpus_matrix.shape[1]} dims)")
print("Row 0 metadata:", {"id": metadata[0]["id"], "stars": metadata[0]["stars"], "text": metadata[0]["text"][:50] + "..."})

Index built: (500, 1536)  (500 reviews x 1536 dims)
Row 0 metadata: {'id': 0, 'stars': 5, 'text': 'dr. goldberg offers everything i look for in a gen...'}


In [35]:
# ─ CLIP 2 · 2 — The search function ────────────────────────────────
def semantic_search(query: str, top_k: int = 5) -> list[dict]:
    """Return the top_k most relevant reviews for a query."""
    # Step 1: Embed the query - SAME model family as the corpus, or the spaces won't match
    response = client.embeddings.create(input=query, model="text-embedding-3-small")
    query_vec = np.array(response.data[0].embedding)

    # Step 2: Vectorized cosine similarity against all docs
    scores = cosine_similarity_matrix(query_vec, corpus_matrix)

    # Step 3: Take the indices of the highest-scoring docs
    top_indices = np.argsort(scores)[::-1][:top_k]

    return [
        {"rank": i + 1, "idx": int(idx), "score": float(scores[idx]),
         "stars": metadata[idx]["stars"], "text": metadata[idx]["text"]}
        for i, idx in enumerate(top_indices)
    ]

print("semantic_search defined.")

semantic_search defined.


In [36]:
# ─ CLIP 2 · 3 — Search that understands meaning ────────────────────
# Semantic search finds relevant reviews even when the query words never appear in them.
for query in ["romantic dinner spot with live music", "quick healthy lunch near downtown"]:
    print(f"\n{'='*60}\nQuery: '{query}'\n{'-'*60}")
    for r in semantic_search(query, top_k=3):
        print(f"  #{r['rank']} [score={r['score']:.4f}, {r['stars']}*] {r['text'][:75]}...")


Query: 'romantic dinner spot with live music'
------------------------------------------------------------
  #1 [score=0.4586, 5*] This location never disappoints!! Food is always consistently great, and if...
  #2 [score=0.4336, 3*] Dined in twice, food ok, atmosphere good....
  #3 [score=0.4288, 5*] Went for dinner Friday evening with friends and sat at the outdoor patio. I...

Query: 'quick healthy lunch near downtown'
------------------------------------------------------------
  #1 [score=0.4661, 2*] This place.. Is ok if ur grabn something quick .. But they do have a great ...
  #2 [score=0.4656, 4*] Went to Rock Bottom for lunch the other day and was very pleased. My cowork...
  #3 [score=0.4346, 3*] If you must eat at Eat N Park, this is the one you want to go to. Why you a...


In [37]:
# ─ CLIP 2 · 4 — The honest limitation ──────────────────────────────
# The honest limitation: search ALWAYS returns its nearest guess, even with no real match.
# There is no ramen review in our 500 - watch the top scores drop as a "nothing fits" signal.
query = "authentic ramen with rich broth"
print(f"Query: '{query}'  (no ramen exists in this corpus)\n{'-'*60}")
for r in semantic_search(query, top_k=3):
    print(f"  #{r['rank']} [score={r['score']:.4f}, {r['stars']}*] {r['text'][:75]}...")
print("\nScores ~0.37 vs ~0.46 earlier -> low top-score means probably nothing relevant.")

Query: 'authentic ramen with rich broth'  (no ramen exists in this corpus)
------------------------------------------------------------
  #1 [score=0.3716, 1*] Delivery is slow and not even close to the best Chinese in the area.  No ma...
  #2 [score=0.3707, 2*] The service was lackluster. Coffee was warm but delicious. Limited to one c...
  #3 [score=0.3697, 3*] We ordered from here because Uncle Chen, our go-to takeout place down the r...

Scores ~0.37 vs ~0.46 earlier -> low top-score means probably nothing relevant.


In [38]:
# ─ CLIP 2 · 5 — Re-ranking (define) ────────────────────────────────
def semantic_search_with_reranking(query, candidate_k=20, final_k=5, star_boost=0.05):
    """Two-stage: retrieve a BROAD candidate set cheaply, then re-rank a NARROW set."""
    candidates = semantic_search(query, top_k=candidate_k)        # retrieve broad
    for r in candidates:                                         # re-rank narrow
        r["reranked_score"] = r["score"] + star_boost * (r["stars"] / 5)
    candidates.sort(key=lambda x: x["reranked_score"], reverse=True)
    return candidates[:final_k]

print("semantic_search_with_reranking defined.")

semantic_search_with_reranking defined.


In [39]:
# ─ CLIP 2 · 6 — Re-ranking (compare) ───────────────────────────────
query = "cozy place for a quiet dinner"
print(f"Query: '{query}'\n\n--- BASE SEARCH (similarity only) ---")
for r in semantic_search(query, top_k=5):
    print(f"  [{r['stars']}*, score={r['score']:.4f}] {r['text'][:65]}...")

print("\n--- RE-RANKED (similarity + star-quality boost) ---")
for r in semantic_search_with_reranking(query, final_k=5):
    print(f"  [{r['stars']}*, final={r['reranked_score']:.4f}] {r['text'][:65]}...")

Query: 'cozy place for a quiet dinner'

--- BASE SEARCH (similarity only) ---
  [3*, score=0.4703] Dined in twice, food ok, atmosphere good....
  [5*, score=0.4573] This is my favorite 'event' restaurant (by which I mean, spendy, ...
  [5*, score=0.4338] Went for dinner Friday evening with friends and sat at the outdoo...
  [2*, score=0.4308] Coming here you get the feeling they haven't changed much.  The t...
  [4*, score=0.4225] Underated and ignored!  if you see this ol time pizzaria near Ken...

--- RE-RANKED (similarity + star-quality boost) ---
  [5*, final=0.5073] This is my favorite 'event' restaurant (by which I mean, spendy, ...
  [3*, final=0.5003] Dined in twice, food ok, atmosphere good....
  [5*, final=0.4838] Went for dinner Friday evening with friends and sat at the outdoo...
  [4*, final=0.4625] Underated and ignored!  if you see this ol time pizzaria near Ken...
  [4*, final=0.4561] My husbands first time.  Nice to be able to pick your fish and st...


---
## Clip 3: Hybrid Search and Quality Evaluation

In [40]:
# ─ CLIP 3 · 1 — Build the BM25 keyword index ───────────────────────
# BM25 works on TOKENS, not vectors: pure literal word-overlap scoring.
print("Building BM25 keyword index...")
tokenized_corpus = [r["text"].lower().split() for r in embedded_reviews]
bm25 = BM25Okapi(tokenized_corpus)
print(f"BM25 index ready: {len(tokenized_corpus)} documents")

Building BM25 keyword index...
BM25 index ready: 500 documents


In [41]:
# ─ CLIP 3 · 2 — Keyword results ────────────────────────────────────
def bm25_search(query: str, top_k: int = 20) -> list[dict]:
    """Pure keyword search. Rare words weigh more than common ones."""
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [{"rank": i + 1, "idx": int(idx), "score": float(scores[idx])}
            for i, idx in enumerate(top_indices)]

print("Top BM25 keyword matches for 'pizza near downtown':")
for r in bm25_search("pizza near downtown", top_k=3):
    print(f"  [bm25={r['score']:.3f}] {metadata[r['idx']]['text'][:75]}...")

Top BM25 keyword matches for 'pizza near downtown':
  [bm25=9.139] I don't care what Cub and Bulls fans think, this Steeler and Penguins fan t...
  [bm25=6.004] We had another opportunity to eat here today and the experience was much im...
  [bm25=5.427] The menu is outstanding but the pizza and beer is too expensive. For 2 pers...


In [42]:
# ─ CLIP 3 · 3 — Reciprocal Rank Fusion ─────────────────────────────
def reciprocal_rank_fusion(semantic_results, bm25_results, k: int = 60, final_k: int = 5):
    """Combine two rankings using RANK only - this sidesteps the score-scale mismatch."""
    rrf_scores = {}
    for rank, result in enumerate(semantic_results, 1):              # contribution from semantic
        rrf_scores[result["idx"]] = rrf_scores.get(result["idx"], 0) + 1 / (k + rank)
    for rank, result in enumerate(bm25_results, 1):                  # contribution from BM25
        rrf_scores[result["idx"]] = rrf_scores.get(result["idx"], 0) + 1 / (k + rank)

    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [{"rank": i + 1, "idx": idx, "rrf_score": score,
             "stars": metadata[idx]["stars"], "text": metadata[idx]["text"]}
            for i, (idx, score) in enumerate(ranked[:final_k])]

print("reciprocal_rank_fusion defined.")

reciprocal_rank_fusion defined.


In [43]:
# ─ CLIP 3 · 4 — Hybrid search in action ────────────────────────────
def hybrid_search(query: str, top_k: int = 5) -> list[dict]:
    """Fuse semantic (meaning) and BM25 (exact keywords) via Reciprocal Rank Fusion."""
    sem_results = semantic_search(query, top_k=20)
    kw_results  = bm25_search(query, top_k=20)
    return reciprocal_rank_fusion(sem_results, kw_results, final_k=top_k)

query = "pizza near downtown"
print(f"Query: '{query}'\n\n--- SEMANTIC ONLY ---")
for r in semantic_search(query, top_k=3):
    print(f"  [score={r['score']:.4f}] {r['text'][:75]}...")
print("\n--- HYBRID (BM25 + Semantic via RRF) ---")
for r in hybrid_search(query, top_k=3):
    print(f"  [rrf={r['rrf_score']:.5f}] {r['text'][:75]}...")

Query: 'pizza near downtown'

--- SEMANTIC ONLY ---
  [score=0.5210] Underated and ignored!  if you see this ol time pizzaria near Kennywood sto...
  [score=0.5189] Okay so I just moved to the outskirts of Duquesne, PA which is outside of P...
  [score=0.4875] All in favor of a deep dish pizza say I!.......IIIIIII,  ok now that i have...

--- HYBRID (BM25 + Semantic via RRF) ---
  [rrf=0.03132] Underated and ignored!  if you see this ol time pizzaria near Kennywood sto...
  [rrf=0.02996] All in favor of a deep dish pizza say I!.......IIIIIII,  ok now that i have...
  [rrf=0.02973] I don't care what Cub and Bulls fans think, this Steeler and Penguins fan t...


In [44]:
# ─ CLIP 3 · 5 — Metadata filtering ─────────────────────────────────
def semantic_search_filtered(query: str, min_stars: int = 4, top_k: int = 5) -> list[dict]:
    """Metadata constraint. Over-retrieve, THEN filter, THEN truncate - or you starve the list."""
    candidates = semantic_search(query, top_k=top_k * 3)            # 3x headroom for the filter
    filtered = [r for r in candidates if r["stars"] >= min_stars]
    return filtered[:top_k]

query = "quiet romantic restaurant"
print(f"Query: '{query}'\n\n--- UNFILTERED ---")
for r in semantic_search(query, top_k=5):
    print(f"  [{r['stars']}*] {r['text'][:65]}...")
print("\n--- FILTERED: 4* and above only ---")
for r in semantic_search_filtered(query, min_stars=4, top_k=5):
    print(f"  [{r['stars']}*] {r['text'][:65]}...")

Query: 'quiet romantic restaurant'

--- UNFILTERED ---
  [5*] This is my favorite 'event' restaurant (by which I mean, spendy, ...
  [2*] Coming here you get the feeling they haven't changed much.  The t...
  [3*] Dined in twice, food ok, atmosphere good....
  [5*] Went for dinner Friday evening with friends and sat at the outdoo...
  [4*] Underated and ignored!  if you see this ol time pizzaria near Ken...

--- FILTERED: 4* and above only ---
  [5*] This is my favorite 'event' restaurant (by which I mean, spendy, ...
  [5*] Went for dinner Friday evening with friends and sat at the outdoo...
  [4*] Underated and ignored!  if you see this ol time pizzaria near Ken...
  [5*] Beautiful decor. Very genuine staff. Very clean washrooms. Delici...
  [4*] My husbands first time.  Nice to be able to pick your fish and st...


### CLIP 3 · 6 — Why evaluate
---
### Evaluating search quality

You can't tune what you can't measure. Evaluation needs **ground truth**: for each test
query, the set of reviews a human judged genuinely relevant. That is exactly what
`relevant_indices` is — a hand-labeled answer key, stored as review **ids**.

How do you build it across 500 reviews? You surface on-topic candidates with a quick
keyword scan, then *read and confirm* them. The next cell shows that derivation, so the
numbers in `eval_set` are real, not guessed.

In [45]:
# ─ CLIP 3 · 7 — Where the ground truth comes from ──────────────────
# HOW WE BUILT THE GROUND TRUTH
# relevant_indices = the review ids a human confirmed as truly on-topic for a query.
# Step 1: surface candidates with a keyword scan.  Step 2 (offline): read them and keep the good ids.
def show_candidates(*keywords, limit=8):
    shown = 0
    for r in embedded_reviews:
        if any(k in r["text"].lower() for k in keywords):
            print(f"  id={r['id']:>3}  {r['stars']}*  {r['text'][:62]}...")
            shown += 1
            if shown == limit:
                break

print("Candidates for 'pizza':")
show_candidates("pizza", "pizzeria")
print("\nCandidates for 'classic diner breakfast':")
show_candidates("diner", "breakfast", "brunch")

Candidates for 'pizza':
  id= 20  4*  A great townie bar with tasty food and an interesting clientel...
  id= 50  1*  The delivery driver mistakenly rang my doorbell, having confus...
  id= 51  3*  3/6/12  visit -  i am from san francisco bay area and if u wan...
  id= 52  1*  Some of the worst pizza I've ever had.  We used a coupon from ...
  id= 69  5*  I brought my husband and my parents all to Papa J's last time ...
  id= 71  5*  What a wonderful surprise found in Carnegie PA, just south of ...
  id= 72  4*  Yay, I'm a fan but sometimes service is a little slow, it was ...
  id= 74  4*  Yay, I'm a fan of the white pizza.  Had take out.  \n\nThe bar...

Candidates for 'classic diner breakfast':
  id= 33  3*  If you want a true understanding of Pittsburgh in the morning,...
  id= 34  5*  Cheap, unpretentious, and, for this, one of my favorite breakf...
  id= 37  3*  OK, what is with all of these \"CASH ONLY\" places in Pittsbur...
  id= 38  5*  BEST DINER IN THE COUNRTY!!! We've been

In [46]:
# ─ CLIP 3 · 8 — The answer key ─────────────────────────────────────
# The answer key, confirmed by reading the candidates above.
# NOTE: in this dataset a review's id equals its row position, so these ids line up
# directly with the idx that semantic_search returns.
eval_set = [
    {"query": "pizza",                   "relevant_indices": [52, 69, 71, 74]},
    {"query": "classic diner breakfast", "relevant_indices": [34, 38, 40, 41, 42]},
    {"query": "chinese takeout",         "relevant_indices": [60, 62, 63]},
    {"query": "great cheeseburger",      "relevant_indices": [19, 23]},
    {"query": "bar with good beer",      "relevant_indices": [20, 21, 35]},
]
print(f"{len(eval_set)} labeled queries ready.")

5 labeled queries ready.


In [47]:
# ─ CLIP 3 · 9 — The three eval metrics ─────────────────────────────
def precision_at_k(retrieved_indices: list[int], relevant_indices: list[int], k: int) -> float:
    """Of the top-k returned, what fraction are relevant? (Did we waste the user's attention?)"""
    top_k = set(retrieved_indices[:k])
    return len(top_k & set(relevant_indices)) / k


def recall_at_k(retrieved_indices: list[int], relevant_indices: list[int], k: int) -> float:
    """Of ALL relevant items, what fraction did we surface in the top-k? (Did we miss things?)"""
    relevant = set(relevant_indices)
    if not relevant:
        return 0.0
    return len(set(retrieved_indices[:k]) & relevant) / len(relevant)


def mean_reciprocal_rank(retrieved_indices: list[int], relevant_indices: list[int]) -> float:
    """1 / rank of the FIRST relevant result. Rewards putting something good near the top."""
    relevant = set(relevant_indices)
    for rank, idx in enumerate(retrieved_indices, 1):
        if idx in relevant:
            return 1.0 / rank
    return 0.0

print("Metrics defined.")

Metrics defined.


In [48]:
# ─ CLIP 3 · 10 — Reading the results & improving ───────────────────
import pandas as pd

K = 5
rows = []
for item in eval_set:
    retrieved = [r["idx"] for r in semantic_search(item["query"], top_k=K)]
    rows.append({
        "query":   item["query"],
        "labeled": len(item["relevant_indices"]),          # how many we marked relevant
        f"P@{K}":  precision_at_k(retrieved, item["relevant_indices"], K),
        f"R@{K}":  recall_at_k(retrieved, item["relevant_indices"], K),
        "MRR":     mean_reciprocal_rank(retrieved, item["relevant_indices"]),
    })

results = pd.DataFrame(rows).round(2)

# Append a MEAN row for the headline numbers
mean_row = {
    "query": "MEAN", "labeled": "",
    f"P@{K}": round(results[f"P@{K}"].mean(), 3),
    f"R@{K}": round(results[f"R@{K}"].mean(), 3),
    "MRR":    round(results["MRR"].mean(), 3),
}
results = pd.concat([results, pd.DataFrame([mean_row])], ignore_index=True)

print(f"semantic_search evaluation  (k={K})")
results


semantic_search evaluation  (k=5)


,query,labeled,P@5,R@5,MRR
0,pizza,4,0.20,0.250,1.00
1,classic diner breakfast,5,0.60,0.600,1.00
2,chinese takeout,3,0.20,0.330,0.20
3,great cheeseburger,2,0.20,0.500,1.00
4,bar with good beer,3,0.00,0.000,0.00
5,MEAN,,0.24,0.336,0.64
